In [8]:
# Used to connect goggle drive to colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Done for checking if drive is connected or not
import os

os.chdir("/content/drive/MyDrive/Local_business project")
print(os.listdir())

['yelp_academic_dataset_business.json', 'yelp_academic_dataset_checkin.json', 'yelp_academic_dataset_review.json', 'yelp_academic_dataset_tip.json', 'yelp_academic_dataset_user.json', 'business_small.json', 'review_small.json']


In [4]:
# Reduced datasize of business.json and kept only neccessary columns in new file business_small.json
import json

business_ids = set()
count = 0
limit = 50000

with open("yelp_academic_dataset_business.json", "r", encoding="utf-8") as fin, open("business_small.json", "w", encoding="utf-8") as fout:
    for line in fin:
        data = json.loads(line)

        categories = data.get("categories", "")
        if categories and ("Food" in categories or "Restaurant" in categories or "Shop" in categories):
            fout.write(json.dumps(data) + "\n")
            business_ids.add(data["business_id"])
            count += 1

        if count >= limit:
            break

print("Businesses saved:", count)

Businesses saved: 50000


In [5]:
# Reduce datasize of review dataset to keep reviews of business_ids that are present in new file
count = 0
limit = 200000

with open("yelp_academic_dataset_review.json", "r", encoding="utf-8") as fin, open("review_small.json", "w", encoding="utf-8") as fout:
    for line in fin:
        data = json.loads(line)

        if data["business_id"] in business_ids:
            fout.write(json.dumps(data) + "\n")
            count += 1

        if count >= limit:
            break

print("Reviews saved:", count)

Reviews saved: 200000


In [6]:
print(os.listdir())

['yelp_academic_dataset_business.json', 'yelp_academic_dataset_checkin.json', 'yelp_academic_dataset_review.json', 'yelp_academic_dataset_tip.json', 'yelp_academic_dataset_user.json', 'business_small.json', 'review_small.json']


In [7]:
# Convert json files that are newly made to csv file for further processing
import pandas as pd

business_df = pd.read_json("business_small.json", lines=True)
review_df = pd.read_json("review_small.json", lines=True)

business_df.to_csv("business_small.csv", index=False)
review_df.to_csv("review_small.csv", index=False)

In [9]:
# used for reading csv files
business_df = pd.read_csv("business_small.csv")
review_df = pd.read_csv("review_small.csv")

In [10]:
# removes rows where important columns like business_id, stars,name have missing values

business_df.dropna(subset=["business_id", "name", "stars"], inplace=True)
review_df.dropna(subset=["business_id", "stars", "text"], inplace=True)

In [11]:
business_df.shape

(50000, 14)

In [12]:
review_df.shape

(200000, 9)

In [14]:
# remove duplicate rows based on unique IDs
business_df.drop_duplicates(subset=["business_id"], inplace=True)
review_df.drop_duplicates(subset=["review_id"], inplace=True)

In [15]:
# Is done to convert reviews in lowercase , remove extra spaces .... isse analysis accurate hoga
review_df["text"] = review_df["text"].str.lower().str.strip()
review_df["text"] = review_df["text"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True)

In [19]:
# This converts star ratings into sentiment labels like positive, neutral and negative
def get_sentiment(stars):
    if stars >= 4:
        return "positive"
    elif stars == 3:
        return "neutral"
    else:
        return "negative"

review_df["sentiment"] = review_df["stars"].apply(get_sentiment)

In [20]:
# yeh new column add krega:- review length (text size) and rating category (high/low
review_df["review_length"] = review_df["text"].apply(len)

review_df["rating_category"] = review_df["stars"].apply(
    lambda x: "high" if x >= 4 else "low"
)

In [23]:
# 3 columns are added in review dataframe :- sentiment, review_length, review_category
review_df.shape

(200000, 12)

In [24]:
# this step will merge two dataframes into one which will be final_df
final_df = review_df.merge(
    business_df[["business_id", "name", "city", "state", "stars", "review_count", "categories"]],
    on="business_id",
    how="left",
    suffixes=("_review", "_business")
)

In [25]:
final_df.shape

(200000, 18)

In [26]:
final_df.columns

Index(['review_id', 'user_id', 'business_id', 'stars_review', 'useful',
       'funny', 'cool', 'text', 'date', 'sentiment', 'review_length',
       'rating_category', 'name', 'city', 'state', 'stars_business',
       'review_count', 'categories'],
      dtype='object')

In [27]:
final_df.dtypes

,0
review_id,object
user_id,object
business_id,object
stars_review,int64
useful,int64
funny,int64
cool,int64
text,object
date,object
sentiment,object


In [28]:
final_df.head()

,review_id,user_id,business_id,stars_review,useful,funny,cool,text,date,sentiment,review_length,rating_category,name,city,state,stars_business,review_count,categories
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,if you decide to eat here just be aware it is ...,2018-07-07 22:09:11,neutral,500,low,Turning Point of North Wales,North Wales,PA,3.0,169,"Restaurants, Breakfast & Brunch, Food, Juice B..."
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,family diner had the buffet eclectic assortmen...,2014-02-05 20:30:30,neutral,323,low,Kettle Restaurant,Tucson,AZ,3.5,47,"Restaurants, Breakfast & Brunch"
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,wow yummy different delicious our favorite...,2015-01-04 00:01:03,positive,226,high,Zaika,Philadelphia,PA,4.0,181,"Halal, Pakistani, Restaurants, Indian"
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,cute interior and owner gave us tour of upcom...,2017-01-14 20:54:15,positive,514,high,Melt,New Orleans,LA,4.0,32,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ..."
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,i am a long term frequent customer of this est...,2015-09-23 23:10:31,negative,327,low,Dmitri's,Philadelphia,PA,4.0,273,"Mediterranean, Restaurants, Seafood, Greek"


In [29]:
final_df.to_csv("final_reviews_cleaned.csv", index=False)